# ETF Momentum — Exploratory Data Analysis

Initial exploration of the cleaned 43-ETF daily panel. Goals:

1. Sanity-check coverage and prices.
2. Visualize the cross-section: cumulative returns and correlations.
3. Compute a baseline 6-month (skip-1) momentum signal and inspect rankings.
4. Run a baseline backtest and compare against SPY and an equal-weight benchmark.

In [ ]:
import sys
from pathlib import Path

# Make the `src/` package importable from the notebook
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)

from src.data import load_panel, load_universe, to_wide
from src.signals import momentum_total_return, cross_sectional_rank
from src.backtest import run_backtest, equal_weight_benchmark
from src.metrics import summary

## 1. Load and inspect

In [ ]:
panel = load_panel()
universe = load_universe()
print(f'Panel: {len(panel):,} rows, {panel["ticker"].nunique()} tickers')
print(f'Date range: {panel["date"].min().date()} -> {panel["date"].max().date()}')
panel.head()

In [ ]:
universe.groupby('category')['ticker'].apply(list)

## 2. Cumulative returns

In [ ]:
closes = to_wide(panel, 'close')
cum_returns = closes / closes.iloc[0] - 1.0

fig, ax = plt.subplots(figsize=(13, 7))
for tkr in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[tkr], alpha=0.5, linewidth=0.8)
ax.set_title('Cumulative return — all 43 ETFs (price-only, no dividends)')
ax.set_ylabel('cumulative return')
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Correlation structure

Daily returns, full sample. Look for clusters — sector groups and regional groups should hang together. Highly correlated names (>0.9) are effectively substitutes, which we may want to deduplicate later.

In [ ]:
rets = to_wide(panel, 'return')
corr = rets.corr()

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Daily-return correlation matrix, 43-ETF universe')
plt.tight_layout()
plt.show()

## 4. Compute baseline momentum signal

Classic 6-month skip-1-month momentum (the Asness-style "6-1" formulation). The skip avoids the well-documented one-month short-term reversal.

In [ ]:
panel['mom6'] = momentum_total_return(panel, lookback_months=6, skip_months=1)
panel['mom6_rank'] = cross_sectional_rank(panel, 'mom6')

latest_date = panel.dropna(subset=['mom6'])['date'].max()
latest = panel[panel['date'] == latest_date].sort_values('mom6', ascending=False)

print(f'Cross-section as of {latest_date.date()}')
print('\nTop 10:')
print(latest.head(10)[['ticker', 'category', 'mom6']].to_string(index=False))
print('\nBottom 10:')
print(latest.tail(10)[['ticker', 'category', 'mom6']].to_string(index=False))

## 5. Baseline backtest

Long top 10 by 6m-skip-1 momentum, monthly rebalance, 5 bps per-side cost. No leverage, no shorts.

In [ ]:
result = run_backtest(
    panel,
    signal_col='mom6',
    n_long=10,
    rebalance='ME',
    cost_bps_per_side=5.0,
)

# Benchmarks
ew_bench = equal_weight_benchmark(panel)
spy = panel[panel['ticker'] == 'spy'].set_index('date')['close']
spy_eq = spy / spy.iloc[0]

# Align all to the backtest window
eq = result.equity_curve
ew_bench_aligned = ew_bench.reindex(eq.index).ffill()
ew_bench_aligned = ew_bench_aligned / ew_bench_aligned.iloc[0]
spy_aligned = spy_eq.reindex(eq.index).ffill()
spy_aligned = spy_aligned / spy_aligned.iloc[0]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(eq.index, eq, label='Momentum (top 10, 6m skip 1)', linewidth=2)
ax.plot(spy_aligned.index, spy_aligned, label='SPY', alpha=0.8)
ax.plot(ew_bench_aligned.index, ew_bench_aligned, label='Equal-weight universe', alpha=0.8)
ax.set_title('Equity curves')
ax.set_ylabel('cumulative return (start = 1.0)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Summary statistics

In [ ]:
# Strategy daily returns
strat_ret = result.daily_returns

# Benchmark daily returns from their equity curves
spy_ret = spy_aligned.pct_change()
ew_ret = ew_bench_aligned.pct_change()

stats = pd.concat([
    summary(strat_ret, 'Momentum 6m-1'),
    summary(spy_ret, 'SPY'),
    summary(ew_ret, 'Equal-weight'),
])

stats.style.format({
    'CAGR': '{:.1%}',
    'Vol (ann.)': '{:.1%}',
    'Sharpe': '{:.2f}',
    'Sortino': '{:.2f}',
    'Max DD': '{:.1%}',
    'Calmar': '{:.2f}',
    'Hit rate': '{:.1%}',
})

## 7. Drawdowns

In [ ]:
def drawdown_series(returns):
    eq = (1 + returns.fillna(0)).cumprod()
    return eq / eq.cummax() - 1

dd_strat = drawdown_series(strat_ret)
dd_spy = drawdown_series(spy_ret)

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(dd_strat.index, dd_strat, 0, alpha=0.4, label='Momentum')
ax.plot(dd_spy.index, dd_spy, color='red', alpha=0.7, label='SPY')
ax.set_title('Drawdowns')
ax.set_ylabel('drawdown')
ax.legend()
plt.tight_layout()
plt.show()

---

## Next steps

1. **Lookback sensitivity** — sweep `lookback_months` over {3, 6, 9, 12} and `skip_months` over {0, 1}, build a Sharpe heatmap.
2. **N sensitivity** — vary `n_long` over {3, 5, 10, 15, 20}.
3. **Risk-adjusted variant** — compare raw 6-1 momentum vs vol-scaled 6-1 momentum.
4. **Walk-forward OOS** — split the sample, choose parameters in-sample, evaluate out-of-sample.
5. **Regime overlay** — gate the strategy based on broad market trend or VIX level.
6. **Turnover and cost sensitivity** — at what cost level does the edge disappear?